# Detecção de Fraude em Cartões de Crédito
Análise de ponta a ponta abordando classificação desbalanceada com XGBoost, ajuste de limiares, avaliação detalhada e interpretabilidade global com SHAP.

## 1. Importações e Exploração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Download direto da base de dados
url = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
df = pd.read_csv(url)

print(f'Dimensões da base: {df.shape}')
display(df.head())

# Proporção das Classes
proporcao = df['Class'].value_counts(normalize=True) * 100
print('\nProporção das classes (%):')
print(proporcao.round(4))

## 2. Preparação e Divisão Estratificada

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df['Log_Amount'] = np.log1p(df['Amount'])

X = df.drop(columns=['Class', 'Amount'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
cols_to_scale = ['Time', 'Log_Amount']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print(f'Fraudes no treino: {y_train.sum()} ({y_train.mean()*100:.3f}%)')
print(f'Fraudes no teste: {y_test.sum()} ({y_test.mean()*100:.3f}%)')

## 3. Treinamento e Comparação de Modelos

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, precision_recall_curve, auc, roc_auc_score

ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

modelos = {
    'Regressão Logística (Baseline)': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest (Balanced)': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, scale_pos_weight=ratio, random_state=42, eval_metric='logloss', n_jobs=-1)
}

resultados = {}

for nome, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)
    y_prob = modelo.predict_proba(X_test_scaled)[:, 1]
    
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(recall, precision)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    resultados[nome] = {'modelo': modelo, 'y_prob': y_prob, 'pr_auc': pr_auc, 'roc_auc': roc_auc}
    
    print(f'=== {nome} ===')
    print(f'ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Fraude (1)']))

## 4. Curva Precisão-Recall vs. Curva ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for nome, dados in resultados.items():
    precision, recall, _ = precision_recall_curve(y_test, dados['y_prob'])
    axes[0].plot(recall, precision, label=f"{nome} (PR-AUC = {dados['pr_auc']:.3f})")

axes[0].set_title('Curva Precisão-Recall (Foco na Classe Fraude)')
axes[0].set_xlabel('Recall (Sensibilidade)')
axes[0].set_ylabel('Precisão')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

from sklearn.metrics import roc_curve
for nome, dados in resultados.items():
    fpr, tpr, _ = roc_curve(y_test, dados['y_prob'])
    axes[1].plot(fpr, tpr, label=f"{nome} (ROC-AUC = {dados['roc_auc']:.3f})")

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('Curva ROC')
axes[1].set_xlabel('Taxa de Falsos Positivos')
axes[1].set_ylabel('Taxa de Verdadeiros Positivos (Recall)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Pipeline, GridSearchCV e Ajuste de Limiar

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

preprocessador = ColumnTransformer(
    transformers=[('num', StandardScaler(), ['Time', 'Log_Amount'])],
    remainder='passthrough'
)

pipe_xgb = Pipeline([
    ('prep', preprocessador),
    ('clf', XGBClassifier(scale_pos_weight=ratio, random_state=42, eval_metric='logloss', n_jobs=-1))
])

param_grid = {
    'clf__max_depth': [3, 5],
    'clf__learning_rate': [0.05, 0.1],
    'clf__n_estimators': [100, 150]
}

grid = GridSearchCV(pipe_xgb, param_grid, cv=3, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

print('Melhores hiperparâmetros:', grid.best_params_)

y_prob_best = grid.predict_proba(X_test)[:, 1]

for limiar in [0.5, 0.3]:
    y_pred_limiar = (y_prob_best >= limiar).astype(int)
    print(f'\n--- Limiar de Decisão: {limiar} ---')
    print(f'Recall: {recall_score(y_test, y_pred_limiar):.4f} | Precisão: {precision_score(y_test, y_pred_limiar):.4f} | F1: {f1_score(y_test, y_pred_limiar):.4f}')
    print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred_limiar))

## 6. Explicabilidade Global com SHAP

In [ ]:
import shap

best_model = grid.best_estimator_.named_steps['clf']
X_test_transformed = grid.best_estimator_.named_steps['prep'].transform(X_test)
feature_names = ['Time', 'Log_Amount'] + [c for c in X_test.columns if c not in ['Time', 'Log_Amount']]
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_df.iloc[:1000])

shap.summary_plot(shap_values, X_test_df.iloc[:1000])